<a href="https://colab.research.google.com/github/FattahulAlim/Machine_Learning_Sem5/blob/main/JS11/TG11_244107020018_MFattahulAlim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import library & mount drive
import numpy as np
import pandas as pd
import cv2
import matplotlib.image as mpimg
from pathlib import Path
from google.colab import drive

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path_voice = '/content/drive/MyDrive/ml-sem5/JS11/dataset/voice.csv'
df = pd.read_csv(path_voice)
print(df.shape)
df.head()

(3168, 21)


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [ ]:
# Memisahkan fitur (X) dan target/label (y)
X = df.drop(columns=['label'])
y = LabelEncoder().fit_transform(df['label'])

print(X.shape, y.shape)
print(pd.Series(y).value_counts())

(3168, 20) (3168,)
1    1584
0    1584
Name: count, dtype: int64


In [ ]:
# Standarisasi fitur
# Bangun model SVM: 3 kernel x 2 rasio split
kernels = ['linear', 'poly', 'rbf']
splits = {'70:30': 0.30, '80:20': 0.20}

hasil = []

for nama_split, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    for k in kernels:
        model = make_pipeline(StandardScaler(), SVC(kernel=k))
        model.fit(X_train, y_train)

        acc_train = accuracy_score(y_train, model.predict(X_train))
        acc_test = accuracy_score(y_test, model.predict(X_test))

        hasil.append({
            'Split': nama_split,
            'Kernel': k,
            'Akurasi Train': round(acc_train, 4),
            'Akurasi Test': round(acc_test, 4)
        })

In [ ]:
# Tabulasi performansi (akurasi)
tabel_voice = pd.DataFrame(hasil)
display(tabel_voice)

# Versi pivot biar gampang dibandingkan
display(tabel_voice.pivot(index='Kernel', columns='Split', values='Akurasi Test'))

,Split,Kernel,Akurasi Train,Akurasi Test
0,70:30,linear,0.9720,0.9790
1,70:30,poly,0.9666,0.9600
2,70:30,rbf,0.9838,0.9832
3,80:20,linear,0.9767,0.9748
4,80:20,poly,0.9680,0.9558
5,80:20,rbf,0.9838,0.9826


Split,70:30,80:20
Kernel,,
linear,0.9790,0.9748
poly,0.9600,0.9558
rbf,0.9832,0.9826


# Tugas 2

In [ ]:
# Load dataset
path_dataset = '/content/drive/MyDrive/ml-sem5/JS11/dataset/'
train_dir = path_dataset + "images/images/training/"
test_dir = path_dataset + "images/images/test/"

def load_dataset(img_dir):
    p = Path(img_dir)
    img_list = []
    for d in p.glob('*'):
        label = str(d).split('/')[-1]
        for file in d.glob('*.jpg'):
            img = mpimg.imread(file)
            if img is not None:
                img_list.append((img, label))
    return img_list

# Gabung train + test dulu, karena nanti di-split ulang 80:20
all_img = load_dataset(train_dir) + load_dataset(test_dir)
print('Total gambar:', len(all_img))

Total gambar: 400


In [ ]:
# Preprocessing: resize + encode label (day = 1, night = 0)
def preprocess(img_list):
    out = []
    for img, label in img_list:
        std_img = cv2.resize(img, (1100, 600))
        out.append((std_img, 1 if label == 'day' else 0))
    return out

all_std = preprocess(all_img)

In [ ]:
# Ekstraksi fitur histogram kanal V (kecerahan) pada ruang warna HSV
# Normalisasi histogram supaya tidak tergantung ukuran gambar
BINS = 32

def hist_feature(image, bins=BINS):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [2], None, [bins], [0, 256]).flatten()
    hist = hist / hist.sum()
    return hist

X_img = np.array([hist_feature(img) for img, _ in all_std])
y_img = np.array([label for _, label in all_std])

print('Shape fitur:', X_img.shape)
print(pd.Series(y_img).value_counts())

Shape fitur: (400, 32)
0    200
1    200
Name: count, dtype: int64


In [ ]:
# Split 80:20
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    X_img, y_img, test_size=0.20, random_state=42, stratify=y_img
)
print(Xi_train.shape, Xi_test.shape)

(320, 32) (80, 32)


In [ ]:
# Baseline: SVM RBF default
base = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
base.fit(Xi_train, yi_train)
acc_base = accuracy_score(yi_test, base.predict(Xi_test))
print(f'Akurasi baseline (C=1, gamma=scale): {acc_base:.4f}')

Akurasi baseline (C=1, gamma=scale): 0.9875


In [ ]:
# Hyperparameter tuning RBF (C dan gamma) pakai GridSearchCV
pipe = make_pipeline(StandardScaler(), SVC(kernel='rbf'))

param_grid = {
    'svc__C': [0.1, 1, 10, 100, 1000],
    'svc__gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(Xi_train, yi_train)

print('Parameter terbaik :', grid.best_params_)
print(f'Akurasi CV terbaik: {grid.best_score_:.4f}')

Parameter terbaik : {'svc__C': 10, 'svc__gamma': 0.1}
Akurasi CV terbaik: 0.9938


In [ ]:
# Evaluasi model terbaik pada data test
best = grid.best_estimator_

acc_train_img = accuracy_score(yi_train, best.predict(Xi_train))
acc_test_img = accuracy_score(yi_test, best.predict(Xi_test))

print(f'Akurasi train (tuned): {acc_train_img:.4f}')
print(f'Akurasi test  (tuned): {acc_test_img:.4f}')

Akurasi train (tuned): 1.0000
Akurasi test  (tuned): 1.0000


In [ ]:
# Catatan performansi nomor 2
tabel_img = pd.DataFrame([
    {'Model': 'SVM RBF default', 'Params': 'C=1, gamma=scale', 'Akurasi Test': round(acc_base, 4)},
    {'Model': 'SVM RBF tuned', 'Params': str(grid.best_params_), 'Akurasi Test': round(acc_test_img, 4)},
])
display(tabel_img)

,Model,Params,Akurasi Test
0,SVM RBF default,"C=1, gamma=scale",0.9875
1,SVM RBF tuned,"{'svc__C': 10, 'svc__gamma': 0.1}",1.0000
